In [3]:
from collections import defaultdict, Counter

In [1]:
def read_sentences_from_file(filename):
    with open(filename, 'r') as file:
        sentences = file.read().splitlines()
    return sentences

# Example usage
filename = 'sentences.txt'
sentences = read_sentences_from_file(filename)

In [2]:
class MarkovModel:
    def __init__(self):
        self.transitions = defaultdict(Counter)

    def train(self, sentences):
        for sentence in sentences:
            words = sentence.split()
            for i in range(len(words) - 1):
                current_word = words[i]
                next_word = words[i + 1]
                self.transitions[current_word][next_word] += 1

    def get_probability(self, context, next_word):
        if context not in self.transitions:
            return 0
        total_transitions = sum(self.transitions[context].values())
        return self.transitions[context][next_word] / total_transitions


In [4]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end_of_word = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.is_end_of_word = True

    def search_prefix(self, prefix):
        node = self.root
        for char in prefix:
            if char not in node.children:
                return None
            node = node.children[char]
        return node

    def autocomplete(self, prefix, markov_model, context):
        node = self.search_prefix(prefix)
        if not node:
            return []

        suggestions = []
        self._dfs(node, prefix, suggestions)
        suggestions.sort(key=lambda word: -markov_model.get_probability(context, word))
        return suggestions

    def _dfs(self, node, prefix, suggestions):
        if node.is_end_of_word:
            suggestions.append(prefix)
        
        for char, next_node in node.children.items():
            self._dfs(next_node, prefix + char, suggestions)


In [9]:
# Read and preprocess sentences
sentences = read_sentences_from_file(filename)

# Train the Markov model
markov_model = MarkovModel()
markov_model.train(sentences)

# Insert words into the Trie
trie = Trie()
for sentence in sentences:
    words = sentence.split()
    for word in words:
        trie.insert(word)

# Autocomplete function
def autocomplete_sentence(prefix, context):
    return trie.autocomplete(prefix, markov_model, context)

# Example usage
prefix = 'ther'
context = 'are'
suggestions = autocomplete_sentence(prefix, context)
print(f"Autocomplete suggestions for prefix '{prefix}' with context '{context}': {suggestions}")


Autocomplete suggestions for prefix 'ther' with context 'are': ['there', 'therapy', 'therapeutic', 'thermostat', 'thermistor']
